# Member 6 — Feature Selection & Dimension Reduction (PCA)

**Technique:** Select informative features (`SelectKBest`) and reduce dimensionality with **PCA**.

## Why this dataset needs it
Engineered histograms are correlated and moderately wide. PCA compresses signal for visualization and can reduce overfitting for small-to-medium image sets.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve Group_Deliverable root whether cwd is notebooks/ or deliverable root
HERE = Path.cwd().resolve()
ROOT = None
for p in (HERE, *HERE.parents):
    if (p / "src" / "preprocess_utils.py").exists():
        ROOT = p
        break
    if (p / "Group_Deliverable" / "src" / "preprocess_utils.py").exists():
        ROOT = p / "Group_Deliverable"
        break
if ROOT is None:
    raise FileNotFoundError("Run from progress/Group_Deliverable or its notebooks/ folder.")

sys.path.insert(0, str(ROOT / "src"))
from preprocess_utils import (
    SEED, SOURCE_URL, DOI, FOLDERS, CLASSES, paths,
    inventory_table, discover_images, audit_images,
    remove_exact_duplicates, iqr_mask, extract_feature_matrix, read_rgb,
    stratified_sample,
)

P = paths(ROOT)
RAW, VIZ, OUT, LOGS = P["raw"], P["viz"], P["outputs"], P["logs"]
for d in (VIZ, OUT, LOGS):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(SEED)
print("Deliverable root:", ROOT)
print("Raw data:", RAW)
print("Dataset:", SOURCE_URL, "| DOI:", DOI)


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler

npz_path = OUT / "m4_scaled_features.npz"
if npz_path.exists():
    data = np.load(npz_path, allow_pickle=True)
    X_raw, y = data["X_raw"], data["y"]
    print("Loaded Member 4 features:", X_raw.shape)
else:
    meta_path = OUT / "m3_cleaned_no_outliers.csv"
    meta = pd.read_csv(meta_path) if meta_path.exists() else audit_images(RAW)[0]
    sample = stratified_sample(meta, 120)
    X_raw, y, _ = extract_feature_matrix(RAW, sample)
    print("Computed features:", X_raw.shape)

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_std = StandardScaler().fit_transform(X_raw)
k = min(20, X_std.shape[1])
selector = SelectKBest(score_func=f_classif, k=k)
X_sel = selector.fit_transform(X_std, y_enc)
print(f"SelectKBest kept {k} / {X_std.shape[1]} features")

pca = PCA(n_components=min(10, X_sel.shape[1]), random_state=SEED)
X_pca = pca.fit_transform(X_sel)
print("Explained variance ratio (first 5):", np.round(pca.explained_variance_ratio_[:5], 4))
print("Cumulative variance (all kept PCs):", float(pca.explained_variance_ratio_.sum()))

np.savez_compressed(
    OUT / "m6_selected_pca_features.npz",
    X_selected=X_sel,
    X_pca=X_pca,
    y=y,
    explained_variance_ratio=pca.explained_variance_ratio_,
)
pd.DataFrame({
    "pc": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative": np.cumsum(pca.explained_variance_ratio_),
}).to_csv(OUT / "m6_pca_variance.csv", index=False)


## EDA visualization — PCA projection & explained variance


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Screes
axes[0].plot(range(1, len(pca.explained_variance_ratio_) + 1), np.cumsum(pca.explained_variance_ratio_), marker="o")
axes[0].set_xlabel("Number of principal components")
axes[0].set_ylabel("Cumulative explained variance")
axes[0].set_title("PCA explained variance")
axes[0].axhline(0.9, color="red", ls="--", lw=1, label="90%")
axes[0].legend()

# 2D scatter
for cls, color in zip(CLASSES, ["#2ca02c", "#d62728"]):
    mask = y == cls
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], s=18, alpha=0.7, label=cls, c=color)
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].set_title("PCA scatter (SelectKBest → PCA)")
axes[1].legend()

fig.tight_layout()
fig.savefig(VIZ / "m6_pca_variance_and_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Interpretation: if Healthy/Unhealthy form soft clusters in PC1–PC2, engineered color features carry separable signal after selection/reduction.")


## Viva talking points
1. Role of SelectKBest vs PCA (filter selection vs rotation/compression).
2. Read the cumulative variance curve.
3. Interpret class separation on the PC1–PC2 scatter.
